# Inspection des patches outliers

**Objectif** : détecter les patches qui s'éloignent le plus du centroïde de leur catégorie dans l'espace des features, et les visualiser pour diagnostiquer :  
- **Annotation bruitée** (patch mal étiqueté)  
- **Frontière ambiguë** (patch à cheval entre deux textures)  
- **Erreur du modèle** (patch correct mais features non discriminantes)

Métrique : **silhouette individuel** = `(b − a) / max(a, b)` où `a` = distance au centroïde de SA catégorie, `b` = distance au centroïde de la catégorie voisine la plus proche.  
Un silhouette **négatif** = le patch est plus proche d'une autre catégorie que de la sienne.

---
## ⚙️ Cellule 1 — Paramètres
**Modifiez les valeurs ci-dessous avant d'exécuter le notebook.**

In [ ]:
# ============================================================
#  PARAMÈTRES — MODIFIER ICI
# ============================================================

# ← CHANGER ICI la catégorie à inspecter (id numérique)
# Catégories disponibles :
#   1 = Totalement homogène
#   3 = Faisceaux
#   4 = Filaments
#   5 = Stratifié rectiligne
#   6 = Stratifié sinueux
#   7 = Granuleux
#   9 = Trou
CATEGORY_TO_INSPECT = 9

# ← CHANGER ICI le block de features à utiliser
# Ex: 'block_4', 'block_8', 'stage_2_fpn', 'stage_3_fpn'
KEY = 'stage_2_fpn'

# ← Nombre de patches outliers à afficher
N_WORST = 5

# ← Chemin vers les images Ouassim brutes
IMG_DIR = 'Image_Ouassim'

# ← Catégories valides (exclure les rares)
CATS_VALID = [1, 3, 4, 5, 6, 7, 9]

# ← Marge de contexte autour du patch (pixels image originale)
CONTEXT_PX = 30

# ← Référence pour "distance à la catégorie" :
#   True  = médoïde  (patch RÉEL le plus représentatif — on peut le visualiser)
#   False = centroïde (moyenne de tous les patches — vecteur abstrait)
USE_MEDOID = True

# Paramètres fixes
SEED    = 42
PCA_DIM = 50

# ============================================================

---
## 📦 Cellule 2 — Imports et chargement
Charge la base HDF5 et le config. Affiche le mapping id → nom des catégories.

In [ ]:
import json
import warnings
from pathlib import Path

import h5py
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
from PIL import Image
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')
np.random.seed(SEED)

# ── Racine du projet ──────────────────────────────────────────────────────────
_out_ROOT    = Path('.').resolve()
_out_IMG_DIR = _out_ROOT / IMG_DIR
_out_OUT_DIR = _out_ROOT / 'output_ouassim' / 'outlier_inspection'
_out_OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Config catégories ─────────────────────────────────────────────────────────
_out_cfg_path = _out_ROOT / 'PatchTagger_Output' / 'config' / 'config.json'
if _out_cfg_path.exists():
    with open(_out_cfg_path) as _f:
        _out_cfg = json.load(_f)
    _out_CATEGORIES = {int(k): v['name']
                       for k, v in _out_cfg['available_categories'].items()}
    _out_CAT_COLORS  = {int(k): v.get('color', '#888888')
                        for k, v in _out_cfg['available_categories'].items()}
else:
    _out_CATEGORIES = {c: f'Cat_{c}' for c in CATS_VALID}
    _out_CAT_COLORS  = {}
    print('⚠ config.json absent — noms génériques utilisés')

print('Catégories disponibles (id → nom) :')
for _cid in sorted(_out_CATEGORIES):
    _mark = ' ← INSPECTÉ' if _cid == CATEGORY_TO_INSPECT else (
            '   (exclu)' if _cid not in CATS_VALID else '')
    print(f'  {_cid:3d} : {_out_CATEGORIES[_cid]}{_mark}')

# ── Charger la base HDF5 ─────────────────────────────────────────────────────
_out_h5_paths = sorted((_out_ROOT / 'data' / 'feature_database').glob('*.h5'))
if not _out_h5_paths:
    raise FileNotFoundError('Aucune base HDF5 trouvée dans data/feature_database/')

# Préférer la base Ouassim si disponible
_out_H5_PATH = next(
    (p for p in _out_h5_paths if 'ouassim' in p.name.lower()),
    _out_h5_paths[0]
)
print(f'\nBase HDF5 chargée : {_out_H5_PATH.name}')

with h5py.File(_out_H5_PATH, 'r') as _h5:
    _out_all_names = np.array([
        n.decode('utf-8') if isinstance(n, (bytes, np.bytes_)) else str(n)
        for n in _h5['metadata/image_names'][:]
    ])
    _out_all_cats  = _h5['metadata/category_ids'][:].astype(int)
    _out_all_pos   = _h5['metadata/positions'][:].astype(float)  # x_min,y_min,x_max,y_max
    _out_avail_keys = list(_h5['features'].keys())

print(f'Patches totaux    : {len(_out_all_cats)}')
print(f'Représentations disponibles : {_out_avail_keys[:8]} ...')

if KEY not in _out_avail_keys:
    raise KeyError(f'KEY={KEY!r} absent. Disponibles : {_out_avail_keys}')
if CATEGORY_TO_INSPECT not in CATS_VALID:
    raise ValueError(f'CATEGORY_TO_INSPECT={CATEGORY_TO_INSPECT} pas dans CATS_VALID={CATS_VALID}')

print(f'\n✓ KEY={KEY!r} trouvé')
print(f'✓ Catégorie inspectée : {CATEGORY_TO_INSPECT} = {_out_CATEGORIES.get(CATEGORY_TO_INSPECT, "?")}')

---
## 🔬 Cellule 3 — Préparation des features
Charge `features[KEY]`, filtre les catégories valides, applique PCA-50d + L2-norm.  
Les indices sont alignés : `_out_X50[i]` correspond à `_out_names[i]`, `_out_pos_v[i]`, `_out_y[i]`.

In [ ]:
# ── Filtrage catégories valides ───────────────────────────────────────────────
_out_mask   = np.isin(_out_all_cats, CATS_VALID)
_out_y      = _out_all_cats[_out_mask]
_out_names  = _out_all_names[_out_mask]
_out_pos_v  = _out_all_pos[_out_mask]   # (N, 4): x_min, y_min, x_max, y_max

with h5py.File(_out_H5_PATH, 'r') as _h5:
    _out_X_raw = _h5['features'][KEY][:].astype(np.float32)[_out_mask]

print(f'Patches valides : {_out_mask.sum()}')
print(f'Dimension features brutes : {_out_X_raw.shape[1]}')

for _c in sorted(CATS_VALID):
    _nc = (_out_y == _c).sum()
    _mark = ' ← INSPECTÉ' if _c == CATEGORY_TO_INSPECT else ''
    print(f'  cat {_c:2d} ({_out_CATEGORIES.get(_c, "?"):22}) : {_nc:4d} patches{_mark}')

# ── PCA-50d ───────────────────────────────────────────────────────────────────
_out_n_comp = min(PCA_DIM, _out_X_raw.shape[1])
_out_pca    = PCA(n_components=_out_n_comp, random_state=SEED)
_out_X50    = _out_pca.fit_transform(_out_X_raw)

# L2-normalisation
_out_norms = np.linalg.norm(_out_X50, axis=1, keepdims=True)
_out_X50n  = _out_X50 / np.where(_out_norms < 1e-8, 1.0, _out_norms)

print(f'\nPCA : {_out_X_raw.shape[1]}d → {_out_n_comp}d')
print(f'Variance expliquée cumulée : {_out_pca.explained_variance_ratio_.sum()*100:.1f}%')

---
## 🎯 Cellule 4 — Détection des outliers

**Référence de distance** (contrôlée par `USE_MEDOID`) :

| Mode | Référence | Avantage |
|------|-----------|----------|
| `USE_MEDOID = True` | **Médoïde** = patch réel qui maximise la similarité cosine moyenne aux autres patches de sa catégorie | Interprétable : c'est une vraie imagette qu'on peut regarder |
| `USE_MEDOID = False` | **Centroïde** = moyenne de tous les vecteurs (vecteur abstrait) | Plus stable statistiquement, mais pas visualisable |

Le **médoïde** est le patch existant le plus "central" : celui qui ressemble le plus à tous les autres de sa catégorie. Si un patch est loin du médoïde, il est atypique.

**Silhouette individuel** = `(b − a) / max(a, b)` ∈ [−1, +1]  
- `a` = distance cosine au vecteur de référence de **SA** catégorie  
- `b` = distance cosine au vecteur de référence de la **catégorie voisine la plus proche**  
- **< 0** → le patch ressemble plus à une autre catégorie qu'à la sienne

In [ ]:
# ── Référence par catégorie : médoïde ou centroïde ───────────────────────────
_out_ref_vecs  = {}   # vecteur de référence pour chaque catégorie
_out_medoid_gi = {}   # indice global du médoïde (None si centroïde)

if USE_MEDOID:
    _out_method = 'médoïde'
    for _c in CATS_VALID:
        _mc   = _out_y == _c
        _Xc   = _out_X50n[_mc]
        _gi_c = np.where(_mc)[0]            # indices globaux de cette catégorie
        # Médoïde = patch qui maximise la similarité cosine moyenne aux autres
        _sim_mat  = _Xc @ _Xc.T            # (Nc, Nc) similarités cosines
        _mean_sim = _sim_mat.mean(axis=1)   # similarité moyenne à tous les autres
        _local_med = int(np.argmax(_mean_sim))
        _out_ref_vecs[_c]  = _Xc[_local_med]
        _out_medoid_gi[_c] = int(_gi_c[_local_med])
    print('Référence : MÉDOÏDE (patch réel le plus central)')
    print()
    for _c in CATS_VALID:
        _gi  = _out_medoid_gi[_c]
        _img = _out_names[_gi]
        _pos = _out_pos_v[_gi]
        print(f'  cat {_c:2d} ({_out_CATEGORIES.get(_c,"?"):22})'
              f' → médoïde global idx={_gi:5d}  img={_img[:35]}')
else:
    _out_method = 'centroïde'
    for _c in CATS_VALID:
        _mc  = _out_y == _c
        _mu  = _out_X50n[_mc].mean(axis=0)
        _mu /= (np.linalg.norm(_mu) + 1e-8)
        _out_ref_vecs[_c]  = _mu
        _out_medoid_gi[_c] = None
    print('Référence : CENTROÏDE (vecteur moyen abstrait)')


def _out_cosine_dist(vec, ref):
    return float(1.0 - np.dot(vec, ref))


# ── Masque + calcul silhouette ────────────────────────────────────────────────
_out_mask_c  = _out_y == CATEGORY_TO_INSPECT
_out_idx_c   = np.where(_out_mask_c)[0]
_out_Xc      = _out_X50n[_out_mask_c]
_out_Nc      = len(_out_idx_c)

if _out_Nc == 0:
    raise ValueError(f'Aucun patch pour la catégorie {CATEGORY_TO_INSPECT}')

print(f'\nCatégorie inspectée : '
      f'{_out_CATEGORIES.get(CATEGORY_TO_INSPECT, "?")} ({_out_Nc} patches)')

_out_silhouettes = np.zeros(_out_Nc)
_out_a_dists     = np.zeros(_out_Nc)
_out_b_dists     = np.zeros(_out_Nc)
_out_nearest_cat = np.zeros(_out_Nc, dtype=int)
_out_other_cats  = [c for c in CATS_VALID if c != CATEGORY_TO_INSPECT]

for _pi, _vec in enumerate(_out_Xc):
    _a      = _out_cosine_dist(_vec, _out_ref_vecs[CATEGORY_TO_INSPECT])
    _b_list = [(c, _out_cosine_dist(_vec, _out_ref_vecs[c]))
               for c in _out_other_cats]
    _b_cat, _b = min(_b_list, key=lambda x: x[1])
    _out_a_dists[_pi]     = _a
    _out_b_dists[_pi]     = _b
    _out_nearest_cat[_pi] = _b_cat
    _out_silhouettes[_pi] = (_b - _a) / (max(_a, _b) + 1e-10)

# Tri : pires en premier
_out_rank_order = np.argsort(_out_silhouettes)
_out_worst_pi   = _out_rank_order[:N_WORST]

# ── Tableau résumé ────────────────────────────────────────────────────────────
print(f'\n{"─"*80}')
print(f'{"#":>3} {"Global idx":>10} {"Image":^40} {"Sil":>7} {"Penche vers":^22}')
print(f'{"─"*80}')
for _rank, _pi in enumerate(_out_rank_order[:N_WORST]):
    _gi   = int(_out_idx_c[_pi])
    _sil  = _out_silhouettes[_pi]
    _ncat = _out_nearest_cat[_pi]
    print(f'{_rank+1:>3} {_gi:>10}  {_out_names[_gi][:38]:40}  {_sil:+.3f}  '
          f'{_out_CATEGORIES.get(_ncat, str(_ncat))}')
print(f'{"─"*80}')
print(f'\nSilhouette moyen   : {_out_silhouettes.mean():+.3f}')
print(f'Silhouette médian  : {np.median(_out_silhouettes):+.3f}')
print(f'Patches sil < 0    : {(_out_silhouettes < 0).sum()} / {_out_Nc} '
      f'({100*(_out_silhouettes < 0).mean():.1f}%)')

---
## 🖼️ Cellule 5 — Visualisation des N_WORST imagettes
Pour chaque patch outlier : crop de l'image brute + marge de contexte + rectangle du patch dessiné.  
**À lire** : le titre indique la catégorie annotée et la catégorie vers laquelle le patch "penche".

In [ ]:
_out_cat_name_insp = _out_CATEGORIES.get(CATEGORY_TO_INSPECT, str(CATEGORY_TO_INSPECT))


def _out_load_crop(gi):
    """Retourne (crop, cx0, cy0, x_min, y_min, x_max, y_max) ou (None, 0,0,x,y,x,y)."""
    _img_name = _out_names[gi]
    _x_min, _y_min, _x_max, _y_max = [int(v) for v in _out_pos_v[gi]]
    _img_path = _out_IMG_DIR / _img_name
    if not _img_path.exists():
        return None, 0, 0, _x_min, _y_min, _x_max, _y_max
    _img_arr = np.array(Image.open(_img_path).convert('L'))
    _H, _W   = _img_arr.shape
    _cx0 = max(0, _x_min - CONTEXT_PX)
    _cy0 = max(0, _y_min - CONTEXT_PX)
    _cx1 = min(_W, _x_max + CONTEXT_PX)
    _cy1 = min(_H, _y_max + CONTEXT_PX)
    return _img_arr[_cy0:_cy1, _cx0:_cx1], _cx0, _cy0, _x_min, _y_min, _x_max, _y_max


def _out_show_patch(ax, gi, border_color='red', title='', subtitle=''):
    """Affiche le crop d'un patch dans ax avec rectangle coloré."""
    _crop, _cx0, _cy0, _x_min, _y_min, _x_max, _y_max = _out_load_crop(gi)
    if _crop is None:
        ax.text(0.5, 0.5, f'Image\nmanquante\n{_out_names[gi][:20]}',
                ha='center', va='center', transform=ax.transAxes, fontsize=7, color='red')
        ax.axis('off')
        return
    ax.imshow(_crop, cmap='gray', vmin=0, vmax=255)
    _rx = _x_min - _cx0
    _ry = _y_min - _cy0
    _rect = Rectangle((_rx, _ry), _x_max - _x_min, _y_max - _y_min,
                       linewidth=2, edgecolor=border_color, facecolor='none')
    ax.add_patch(_rect)
    if title:
        ax.set_title(title, fontsize=7.5, pad=3)
    if subtitle:
        ax.text(0.02, 0.02, subtitle, transform=ax.transAxes, fontsize=5.5,
                color='yellow', va='bottom',
                bbox=dict(fc='black', alpha=0.5, pad=1.5))
    ax.axis('off')


# ── Layout : 1 ligne par outlier, colonnes selon USE_MEDOID ──────────────────
_out_ncols_viz = 3 if USE_MEDOID else 1

_out_fig_crops, _out_axes_crops = plt.subplots(
    N_WORST, _out_ncols_viz,
    figsize=(4.5 * _out_ncols_viz, 4.5 * N_WORST),
    squeeze=False
)

_out_suptitle = (
    f'Top-{N_WORST} outliers — {_out_cat_name_insp} — [{KEY}]  ({_out_method})\n'
    + ('  Rouge = outlier  |  Bleu = médoïde catégorie propre  |  Orange = médoïde catégorie cible'
       if USE_MEDOID else '  Rouge = patch annoté')
)
_out_fig_crops.suptitle(_out_suptitle, fontsize=10, fontweight='bold')

for _rank, _pi in enumerate(_out_worst_pi):
    _gi        = int(_out_idx_c[_pi])
    _sil       = _out_silhouettes[_pi]
    _ncat      = int(_out_nearest_cat[_pi])
    _ncat_name = _out_CATEGORIES.get(_ncat, str(_ncat))
    _img_name  = _out_names[_gi]
    _pos_str   = f'({int(_out_pos_v[_gi][0])},{int(_out_pos_v[_gi][1])})'

    # Colonne 0 — l'outlier lui-même
    _out_show_patch(
        _out_axes_crops[_rank, 0], _gi,
        border_color='red',
        title=f'#{_rank+1}  sil={_sil:+.2f}\n'
              f'Annoté : {_out_cat_name_insp[:18]}\n'
              f'→ penche : {_ncat_name[:18]}',
        subtitle=f'{_img_name[:22]}  {_pos_str}'
    )

    if USE_MEDOID:
        # Colonne 1 — médoïde de la catégorie propre
        _med_own_gi = _out_medoid_gi[CATEGORY_TO_INSPECT]
        _out_show_patch(
            _out_axes_crops[_rank, 1], _med_own_gi,
            border_color='dodgerblue',
            title=f'Médoïde — {_out_cat_name_insp[:20]}\n(référence "normal")',
            subtitle=_out_names[_med_own_gi][:28]
        )

        # Colonne 2 — médoïde de la catégorie cible
        _med_tgt_gi = _out_medoid_gi[_ncat]
        _out_show_patch(
            _out_axes_crops[_rank, 2], _med_tgt_gi,
            border_color='darkorange',
            title=f'Médoïde — {_ncat_name[:20]}\n(catégorie cible)',
            subtitle=_out_names[_med_tgt_gi][:28]
        )

_out_fig_crops.tight_layout(h_pad=2.5, w_pad=1.5)
_out_save_crops = _out_OUT_DIR / f'outliers_crops_cat{CATEGORY_TO_INSPECT}_{KEY}.png'
_out_fig_crops.savefig(_out_save_crops, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {_out_save_crops.name}')

if USE_MEDOID:
    print('\nLecture rapide :')
    print('  Col. 1 [rouge]   : le patch outlier')
    print('  Col. 2 [bleu]    : médoïde de SA catégorie — ce que la catégorie ressemble normalement')
    print('  Col. 3 [orange]  : médoïde de la catégorie CIBLE — ce vers quoi le modèle le range')
    print('  → Si le patch ressemble visuellement + à la col. 3 qu\'à la col. 2 : outlier confirmé.')

---
## 🗺️ Cellule 6 — Carte spatiale des outliers
Sur chaque image contenant des patches de la catégorie inspectée, dessine :  
- 🟢 **Vert** : patches bien placés (silhouette > 0)  
- 🔴 **Rouge** : outliers (silhouette < 0)

**À lire** : si les outliers se concentrent aux **frontières** entre zones de textures → problème d'annotation de frontière, pas d'erreur aléatoire.

In [ ]:
# Images contenant au moins 1 patch de la catégorie inspectée
_out_imgs_with_cat = np.unique(_out_names[_out_mask_c])

# Limiter à 6 images max pour lisibilité
_out_imgs_to_show = _out_imgs_with_cat[:6]
_out_ncols = min(3, len(_out_imgs_to_show))
_out_nrows = int(np.ceil(len(_out_imgs_to_show) / _out_ncols))

_out_fig_sp, _out_axes_sp = plt.subplots(
    _out_nrows, _out_ncols,
    figsize=(6 * _out_ncols, 5 * _out_nrows),
    squeeze=False
)

_out_fig_sp.suptitle(
    f'Carte spatiale — {_out_cat_name_insp} [{KEY}]\n'
    'Vert = silhouette≥0 (OK) · Rouge = silhouette<0 (outlier)',
    fontsize=12, fontweight='bold'
)

for _si, _img_name in enumerate(_out_imgs_to_show):
    _ax = _out_axes_sp[_si // _out_ncols, _si % _out_ncols]

    _img_path = _out_IMG_DIR / _img_name
    if not _img_path.exists():
        _ax.text(0.5, 0.5, f'Image manquante:\n{_img_name}',
                 ha='center', va='center', transform=_ax.transAxes)
        _ax.axis('off')
        continue

    _img_arr = np.array(Image.open(_img_path).convert('L'))
    _ax.imshow(_img_arr, cmap='gray', vmin=0, vmax=255)

    # Tous les patches de la catégorie dans cette image
    for _pi_local, _pi_global in enumerate(_out_rank_order):
        # _pi_global est l'index dans _out_idx_c (pour la catégorie inspectée)
        _gi = int(_out_idx_c[_pi_global])
        if _out_names[_gi] != _img_name:
            continue
        _x_min, _y_min, _x_max, _y_max = [int(v) for v in _out_pos_v[_gi]]
        _sil = _out_silhouettes[_pi_global]
        _col = 'red' if _sil < 0 else 'lime'
        _lw  = 2.5 if _sil < 0 else 1.5
        _rect = Rectangle((_x_min, _y_min), _x_max - _x_min, _y_max - _y_min,
                           linewidth=_lw, edgecolor=_col, facecolor='none',
                           alpha=0.85)
        _ax.add_patch(_rect)

    # Tous les autres patches (catégories différentes) — gris discret
    _m_other_img = (_out_names == _img_name) & (_out_y != CATEGORY_TO_INSPECT)
    for _gi_o in np.where(_m_other_img)[0]:
        _x_min, _y_min, _x_max, _y_max = [int(v) for v in _out_pos_v[_gi_o]]
        _rect = Rectangle((_x_min, _y_min), _x_max - _x_min, _y_max - _y_min,
                           linewidth=1, edgecolor='grey', facecolor='none',
                           alpha=0.35)
        _ax.add_patch(_rect)

    _ax.set_title(_img_name[:45], fontsize=8)
    _ax.axis('off')

# Masquer axes vides
for _si in range(len(_out_imgs_to_show), _out_nrows * _out_ncols):
    _out_axes_sp[_si // _out_ncols, _si % _out_ncols].axis('off')

# Légende
_leg = [mpatches.Patch(color='lime',  label='OK (sil≥0)'),
        mpatches.Patch(color='red',   label='Outlier (sil<0)'),
        mpatches.Patch(color='grey',  label='Autre catégorie', alpha=0.35)]
_out_fig_sp.legend(handles=_leg, loc='lower center', ncol=3, fontsize=10,
                    bbox_to_anchor=(0.5, -0.02))

_out_fig_sp.tight_layout()
_out_save_sp = _out_OUT_DIR / f'spatial_map_cat{CATEGORY_TO_INSPECT}_{KEY}.png'
_out_fig_sp.savefig(_out_save_sp, dpi=130, bbox_inches='tight')
plt.show()
print(f'Saved → {_out_save_sp.name}')

---
## 📊 Cellule 7 — Impact quantitatif du retrait des outliers
Calcule la silhouette moyenne de la catégorie **avant** et **après** retrait des X% pires patches.  
Si le score remonte fortement → les outliers plombent vraiment la séparabilité.

In [ ]:
_out_pcts_to_test = [0, 5, 10, 20, 30]

def _out_silhouette_after_removal(pct_remove):
    """Silhouette moyen de la catégorie après retrait des pct_remove % pires."""
    _n_remove = int(np.ceil(_out_Nc * pct_remove / 100))
    # Garder tous sauf les _n_remove pires
    _keep_pi  = _out_rank_order[_n_remove:]   # indices dans _out_idx_c
    if len(_keep_pi) == 0:
        return np.nan
    return float(_out_silhouettes[_keep_pi].mean())


print(f'Catégorie : {_out_cat_name_insp}  |  {_out_Nc} patches au total')
print(f'{"─"*55}')
print(f'{"Retrait (%)":>12} {"Patches restants":>17} {"Silhouette moyen":>18}')
print(f'{"─"*55}')

_out_impact_pcts = []
_out_impact_vals = []

for _pct in _out_pcts_to_test:
    _n_rm  = int(np.ceil(_out_Nc * _pct / 100))
    _n_kp  = _out_Nc - _n_rm
    _sil_m = _out_silhouette_after_removal(_pct)
    _out_impact_pcts.append(_pct)
    _out_impact_vals.append(_sil_m)
    _marker = ' ← baseline' if _pct == 0 else ''
    print(f'{_pct:>12}% {_n_kp:>17d} {_sil_m:>+17.4f}{_marker}')

print(f'{"─"*55}')

# Plot
_out_fig_imp, _out_ax_imp = plt.subplots(figsize=(7, 4))
_out_ax_imp.plot(_out_impact_pcts, _out_impact_vals,
                  'o-', color='#e74c3c', linewidth=2, markersize=8)
_out_ax_imp.axhline(0, color='grey', linestyle='--', linewidth=1)
_out_ax_imp.fill_between(_out_impact_pcts, 0, _out_impact_vals,
                          where=[v > 0 for v in _out_impact_vals],
                          alpha=0.15, color='green', label='sil > 0')
_out_ax_imp.set_xlabel('% patches les plus outliers retirés', fontsize=11)
_out_ax_imp.set_ylabel('Silhouette moyen', fontsize=11)
_out_ax_imp.set_title(
    f'Impact du retrait des outliers — {_out_cat_name_insp}\n'
    f'[{KEY}]  ({_out_Nc} patches)',
    fontsize=11, fontweight='bold'
)
_out_ax_imp.grid(alpha=0.3)
_out_ax_imp.legend(fontsize=9)
_out_fig_imp.tight_layout()
_out_save_imp = _out_OUT_DIR / f'impact_removal_cat{CATEGORY_TO_INSPECT}_{KEY}.png'
_out_fig_imp.savefig(_out_save_imp, dpi=150)
plt.show()
print(f'Saved → {_out_save_imp.name}')

# Résumé
_out_gain = _out_impact_vals[-1] - _out_impact_vals[0]
print(f'\nGain en retirant {_out_pcts_to_test[-1]}% : '
      f'{_out_impact_vals[0]:+.4f} → {_out_impact_vals[-1]:+.4f} '
      f'(Δ={_out_gain:+.4f})')
if _out_gain > 0.05:
    print('→ Les outliers plombent significativement la catégorie.')
elif _out_gain > 0.01:
    print('→ Impact modéré des outliers.')
else:
    print('→ Impact faible : le problème vient du modèle, pas des annotations.')

---
## 📈 Cellule 8 — Distribution des silhouettes
Histogramme de toutes les silhouettes pour la catégorie inspectée.

In [ ]:
_out_fig_dist, (_out_ax_dist, _out_ax_pie) = plt.subplots(1, 2, figsize=(12, 4))

# Histogramme
_out_bins = np.linspace(-1, 1, 30)
_out_ax_dist.hist(_out_silhouettes[_out_silhouettes < 0], bins=_out_bins,
                   color='red', alpha=0.7, label='Outliers (sil<0)')
_out_ax_dist.hist(_out_silhouettes[_out_silhouettes >= 0], bins=_out_bins,
                   color='lime', alpha=0.7, label='OK (sil≥0)')
_out_ax_dist.axvline(0, color='black', linewidth=1.5, linestyle='--')
_out_ax_dist.axvline(_out_silhouettes.mean(), color='orange', linewidth=1.5,
                      linestyle=':', label=f'Moy={_out_silhouettes.mean():+.3f}')
_out_ax_dist.set_xlabel('Silhouette individuel', fontsize=11)
_out_ax_dist.set_ylabel('Nombre de patches', fontsize=11)
_out_ax_dist.set_title(f'Distribution silhouettes — {_out_cat_name_insp}', fontsize=11)
_out_ax_dist.legend(fontsize=9)
_out_ax_dist.grid(alpha=0.3)

# Pie : répartition OK / outliers
_out_n_ok  = (_out_silhouettes >= 0).sum()
_out_n_bad = (_out_silhouettes < 0).sum()
_out_ax_pie.pie(
    [_out_n_ok, _out_n_bad],
    labels=[f'OK ({_out_n_ok})', f'Outliers ({_out_n_bad})'],
    colors=['lime', 'red'], autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
_out_ax_pie.set_title(f'Part d\'outliers\n{_out_cat_name_insp}', fontsize=11)

# Répartition des catégories cibles
print('\nVers quelles catégories penchent les outliers (sil<0) ?')
_out_bad_pi    = _out_rank_order[_out_silhouettes[_out_rank_order] < 0]
_out_bad_ncats = _out_nearest_cat[_out_bad_pi]
for _nc in sorted(set(_out_bad_ncats)):
    _cnt = (_out_bad_ncats == _nc).sum()
    print(f'  → {_out_CATEGORIES.get(_nc, str(_nc)):22}: {_cnt:3d} patches '
          f'({100*_cnt/max(1,len(_out_bad_ncats)):.1f}%)')

_out_fig_dist.tight_layout()
_out_save_dist = _out_OUT_DIR / f'distribution_sil_cat{CATEGORY_TO_INSPECT}_{KEY}.png'
_out_fig_dist.savefig(_out_save_dist, dpi=150)
plt.show()
print(f'Saved → {_out_save_dist.name}')

---
## 🔍 Comment interpréter les résultats ?

### Silhouette < 0 : 3 explications possibles

| Observation visuelle du crop | Diagnostic | Action suggérée |
|------------------------------|------------|----------------|
| **Patch visuellement ambigu** (texture mixte, chevauchement) | Annotation de frontière — le patch est à cheval entre deux zones | Resserrer l'annotation, ou ajouter une catégorie "transition" |
| **Patch net mais annoté dans la mauvaise catégorie** | Annotation incorrecte (erreur humaine) | Ré-étiqueter le patch |
| **Patch visuellement correct** (correspond bien à sa catégorie) | Modèle non discriminant pour cette texture | Pas d'annotation à corriger — explorer d'autres représentations (blocks/FPN) |

### Interprétation de la carte spatiale (cellule 6)

- **Outliers aux frontières** des zones → problème systématique de délimitation → réviser le protocole d'annotation (exclure les patches à moins de N px d'une autre texture).
- **Outliers distribués aléatoirement** dans la zone → annotations individuelles incorrectes → vérifier patch par patch.
- **Outliers dans une seule image** → problème de qualité de cette image (flou, artefact) → exclure l'image.

### Interprétation de l'impact quantitatif (cellule 7)

- **Gain > 0.05** après retrait des 20% pires → les outliers plombent significativement. Vaut la peine de les corriger.
- **Gain ≈ 0** → le problème vient du modèle (domain shift, résolution des features insuffisante), pas des annotations.

---
*Fichiers sauvegardés dans `output_ouassim/outlier_inspection/`*